# Using SNOMED CT terminology server as tool in agentic RAG

In [12]:
from dotenv import load_dotenv

In [13]:
_ = load_dotenv("../.env")

### Calling relation_extractor remote runnable

In [14]:
from langserve.client import RemoteRunnable

In [2]:
relation_extractor_chain = RemoteRunnable("http://localhost:8000/relation_extractor_chain")

In [16]:
test_note = """patient had an acute severe heart attack"""

In [18]:
relation_extractor_chain.invoke({"note": test_note})

{'values': [{'node_1': 'heart attack | 22298006',
   'node_2': 'acute',
   'edge': 'acute severity'},
  {'node_1': 'heart attack | 22298006',
   'node_2': 'severe',
   'edge': 'severe severity'}],
 'context': {'extracted_entities': {'heart attack': {'cui': '22298006',
    'start': 28,
    'end': 40}}}}

In [30]:
test_note[55:63]

'swelling'

### Using SNOMED server APIs

In [15]:
import requests

In [158]:
url = "https://snowstorm-training.snomedtools.org/snowstorm/snomed-ct/MAIN%2F2020-07-31/concepts"

Snowstorm server is unreliable - look for better alternatives, or self host etc...

In [17]:
snowstorm_url = "https://snowstorm.ihtsdotools.org/snowstorm/snomed-ct/MAIN%2F2024-10-01/concepts"

In [18]:
params = {
    "term": "fracture of left femur",
    "includeLeafFlag": "false",
    "form": "inferred",
    "offset": "0",
    "limit": "50"
}

headers = {
    "accept": "application/json",
    "Accept-Language": "en-X-900000000000509007,en-X-900000000000508004,en"
}

In [19]:
response = requests.get(url, params=params, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Request failed with status code: {response.status_code}")
    print(response.text)

{'items': [{'conceptId': '15911931000119102', 'active': True, 'definitionStatus': 'FULLY_DEFINED', 'moduleId': '900000000000207008', 'effectiveTime': '20200131', 'fsn': {'term': 'Open fracture of head of left femur (disorder)', 'lang': 'en'}, 'pt': {'term': 'Open fracture of head of left femur', 'lang': 'en'}, 'id': '15911931000119102'}, {'conceptId': '15912331000119106', 'active': True, 'definitionStatus': 'FULLY_DEFINED', 'moduleId': '900000000000207008', 'effectiveTime': '20190731', 'fsn': {'term': 'Closed fracture of head of left femur (disorder)', 'lang': 'en'}, 'pt': {'term': 'Closed fracture of head of left femur', 'lang': 'en'}, 'id': '15912331000119106'}, {'conceptId': '10820181000119103', 'active': True, 'definitionStatus': 'FULLY_DEFINED', 'moduleId': '900000000000207008', 'effectiveTime': '20190731', 'fsn': {'term': 'Open fracture of distal end of left femur (disorder)', 'lang': 'en'}, 'pt': {'term': 'Open fracture of distal end of left femur', 'lang': 'en'}, 'id': '1082018

In [20]:
for i in range(len(data['items'])):
    print(data['items'][i]['fsn']['term'])

Open fracture of head of left femur (disorder)
Closed fracture of head of left femur (disorder)
Open fracture of distal end of left femur (disorder)
Closed fracture of trochanter of left femur (disorder)
Open fracture of base of neck of left femur (disorder)
Closed reduction of fracture of left femur and internal fixation using dynamic hip screw plate (procedure)


In [177]:
class SnomedClient:
    def __init__(self, url: str):
        self.url = url
        self.headers = {
            "accept": "application/json",
            "Accept-Language": "en-X-900000000000509007,en-X-900000000000508004,en"
        }
        
    def search(self, term: str, limit: int = 5):
        params = {
            "term": term,
            "includeLeafFlag": "false",
            "form": "inferred",
            "offset": "0",
            "limit": limit,
            "termActive": "true",
            "activeFilter": "true",
        }
        response = requests.get(self.url, params=params, headers=self.headers)

        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()
        else:
            raise Exception(f"Request failed with status code: {response.status_code}")
            
        return data
        
    def search_concept_id(self, conceptId: str):
        browser_url = self.url.replace('/snomed-ct/', '/snomed-ct/browser/')
        request_url = f"{browser_url}/{conceptId}"
        print(request_url)
        response = requests.get(request_url, headers=self.headers)

        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()
        else:
            raise Exception(f"Request failed with status code: {response.status_code}")
        return data

In [178]:
url

'https://snowstorm-training.snomedtools.org/snowstorm/snomed-ct/MAIN%2F2020-07-31/concepts'

In [179]:
snomed_client = SnomedClient(url)

In [199]:
snomed_client.search("acute myocardial infarction", limit=3)

{'items': [{'conceptId': '57054005',
   'active': True,
   'definitionStatus': 'FULLY_DEFINED',
   'moduleId': '900000000000207008',
   'effectiveTime': '20020131',
   'fsn': {'term': 'Acute myocardial infarction (disorder)', 'lang': 'en'},
   'pt': {'term': 'Acute myocardial infarction', 'lang': 'en'},
   'id': '57054005'},
  {'conceptId': '304914007',
   'active': True,
   'definitionStatus': 'PRIMITIVE',
   'moduleId': '900000000000207008',
   'effectiveTime': '20020131',
   'fsn': {'term': 'Acute Q wave myocardial infarction (disorder)',
    'lang': 'en'},
   'pt': {'term': 'Acute Q wave myocardial infarction', 'lang': 'en'},
   'id': '304914007'},
  {'conceptId': '58612006',
   'active': True,
   'definitionStatus': 'FULLY_DEFINED',
   'moduleId': '900000000000207008',
   'effectiveTime': '20020731',
   'fsn': {'term': 'Acute myocardial infarction of lateral wall (disorder)',
    'lang': 'en'},
   'pt': {'term': 'Acute myocardial infarction of lateral wall', 'lang': 'en'},
   'id'

In [181]:
full_concept = snomed_client.search_concept_id(57054005)
extract_relations(full_concept)

https://snowstorm-training.snomedtools.org/snowstorm/snomed-ct/browser/MAIN%2F2020-07-31/concepts/57054005


{'INFERRED_RELATIONSHIP': {'Is a': {'id': '116680003',
   'targets': {'Acute ischemic heart disease': '32598000',
    'Myocardial infarction': '22298006',
    'Acute heart disease': '127337006'}},
  'Clinical course': {'id': '263502005',
   'targets': {'Sudden onset AND/OR short duration': '424124008'}},
  'Finding site': {'id': '363698007',
   'targets': {'Myocardium structure': '74281007',
    'Cardiac internal structure': '277712000',
    'Coronary artery structure': '41801008'}},
  'Associated morphology': {'id': '116676008',
   'targets': {'Acute infarct': '55470003',
    'Arteriosclerosis': '28960008',
    'Atherosclerosis': '38716007'}},
  'Course': {'id': '260908002', 'targets': {'Acute': '53737009'}},
  'Onset': {'id': '246100006',
   'targets': {'Acute onset': '373933003', 'Sudden': '255363002'}}},
 'STATED_RELATIONSHIP': {'Is a': {'id': '116680003',
   'targets': {'Acute heart disease': '127337006',
    'Myocardial infarction': '22298006'}},
  'Finding site': {'id': '3636980

### Using LangGraph

In [23]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List, Dict
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage

In [254]:
class AgentState(TypedDict):
    relations: str
    search_terms: List[str]
    snomed_candidates: List[Dict]
    target_attributes: str
    evals: List[Dict]
    shortlist: List[Dict]
    final_candidate: str
    revision_number: int
    max_revisions: int

In [26]:
from langchain_openai import ChatOpenAI

In [27]:
model = ChatOpenAI(model="gpt-4o", temperature=0)

In [55]:
from langchain_community.llms import Replicate
model = Replicate(
    model= "mistralai/mixtral-8x7b-instruct-v0.1:5d78bcd7a992c4b793465bcdcf551dc2ab9668d12bb7aa714557a21c1e77041c",
    model_kwargs={"temperature": 0},
)

In [73]:
QUERY_PROMPT = """You are tasked with creating multiple search terms for a SNOMED CT database based on a given relation triplet. The triplet is provided in the following format:

```
{
  'node_1': [primary concept],
  'node_2': [modifier or related concept],
  'edge': [relationship between node_1 and node_2]
}
```

Your task is to:

1. Analyze the information in the triplet.
2. Generate a list of 1-3 search terms with varying complexity, from simple to more clinically specific.
3. For each term, determine the most appropriate order and combination of 'node_1' and 'node_2'.
4. Consider using clinically appropriate synonyms or more formal medical terminology where applicable.

Rules:
- Each search term should be concise and clinically accurate.
- Generally, place modifiers (like severity or temporality) before the main concept.
- Omit the 'edge' information unless it's crucial for disambiguation.
- Use lowercase unless uppercase is standard for a particular term.
- When providing multiple terms, aim for a range of complexity or specificity.

Example input:
```
{
  'node_1': 'heart attack',
  'node_2': 'acute',
  'edge': 'acute severity'
}
```

Example output:
1. "acute heart attack"
2. "acute myocardial infarction"
3. "acute MI"

Explanation:
1. Uses the common term, directly combining the given nodes.
2. Employs the more formal clinical term "myocardial infarction".
3. Provides an abbreviated form commonly used in clinical settings.

Now, given a relation triplet in the specified format, generate 1-3 appropriate SNOMED CT search terms with varying complexity or specificity."""

In [89]:
ATTRIBUTE_PROMPT = """
You are tasked with analyzing a SNOMED-like medical term and predicting its morphology (the form or structure of an abnormality) and finding site (the bodily location). Your analysis should be based on the structure and components of the given term.

Input:
You will be given a single SNOMED-like term, which may include multiple words.

Task:
1. Analyze the given term to identify its key components.
2. Predict the most likely morphology (form of abnormality or disease process).
3. Predict the most likely finding site (anatomical location).

Output:
Provide your predictions in the following format:
- Term: [The original input term]
- Predicted Morphology: [Your morphology prediction]
- Predicted Finding Site: [Your finding site prediction]

Guidelines:
1. Morphology often relates to the type of pathological process or abnormality (e.g., inflammation, neoplasm, fracture).
2. Finding site typically refers to an anatomical structure or location.
3. Some terms may clearly indicate both morphology and site, while others might imply one more strongly than the other.
4. If a component is not clearly indicated or cannot be reasonably inferred, state "Not specified" for that component.

Examples:

Input: "Acute myocardial infarction"
Output:
- Term: Acute myocardial infarction
- Predicted Morphology: Acute infarction
- Predicted Finding Site: Myocardium

Input: "Chronic obstructive pulmonary disease"
Output:
- Term: Chronic obstructive pulmonary disease
- Predicted Morphology: Narrowing
- Predicted Finding Site: Tracheobronchial

Input: "Melanoma"
Output:
- Term: Melanoma
- Predicted Morphology: Malignant neoplasm
- Predicted Finding Site: Skin (most common, but not specified in the term)

Now, given a SNOMED-like term, predict its morphology and finding site based on these guidelines.
"""

In [229]:
EVALUATION_PROMPT = """
You are tasked with evaluating the appropriateness of a candidate SNOMED CT term based on its predicted attributes and retrieved relationships. You will be given the following information:

1. A candidate SNOMED CT term and its ID
2. Predicted attributes for the term
3. Retrieved relationships from the SNOMED CT server (both inferred and stated)

Your task is to:

1. Analyze the given information.
2. Compare the predicted attributes with the retrieved relationships.
3. Evaluate the appropriateness of the candidate term on a scale of 1-5, where:
   1 = Not appropriate at all
   2 = Somewhat inappropriate
   3 = Moderately appropriate
   4 = Very appropriate
   5 = Perfectly appropriate

Input Format:

```
<candidate>
'[Candidate Term]': '[SNOMED CT ID]'

<predicted_attributes>
{
  "finding_site": "[Predicted Site]",
  "morphology": "[Predicted Morphology]"
}

<retrieved_relationships>
{
  'INFERRED_RELATIONSHIP': {
    '[Relationship Type]': {
      'id': '[Relationship ID]',
      'targets': {
        '[Target Term]': '[Target ID]',
        ...
      }
    },
    ...
  },
  'STATED_RELATIONSHIP': {
    '[Relationship Type]': {
      'id': '[Relationship ID]',
      'targets': {
        '[Target Term]': '[Target ID]',
        ...
      }
    },
    ...
  }
}
```

Evaluation Criteria:
1. Check if the predicted finding site matches or is closely related to any of the "Finding site" targets in the relationships.
2. Check if the predicted morphology matches or is closely related to any of the "Associated morphology" targets in the relationships.
3. Examine other relationships (e.g., "Is a", "Clinical course") to ensure they are consistent with the candidate term.
4. Consider both inferred and stated relationships, giving slightly more weight to stated relationships.

Output Format:
Provide your evaluation in the following format:

score: [1-5]
reasoning:
- [Key point supporting your score]
- [Another key point supporting your score]
- [Any discrepancies or concerns]

Now, given the candidate term, predicted attributes, and retrieved relationships, evaluate the appropriateness of the SNOMED CT candidate term. Keep your reasoning concise.
"""

In [91]:
from pydantic import BaseModel

class Queries(BaseModel):
    queries: List[str]

In [92]:
class Attributes(BaseModel):
    finding_site: str
    morphology: str

In [231]:
class ScoreCard(BaseModel):
    candidate_term: str
    snomed_id: str
    score: int
    reasoning: str

In [131]:
def extract_relations(concept):
    relations = {
        "INFERRED_RELATIONSHIP": {},
        "STATED_RELATIONSHIP": {}
    }
    
    for relationship in concept.get('relationships', []):
        char_type = relationship.get('characteristicType')
        if char_type in ["INFERRED_RELATIONSHIP", "STATED_RELATIONSHIP"]:
            type_info = relationship.get('type', {})
            target_info = relationship.get('target', {})
            
            type_term = type_info.get('pt', {}).get('term')
            type_id = type_info.get('conceptId')
            target_term = target_info.get('pt', {}).get('term')
            target_id = target_info.get('conceptId')
            
            if type_term and type_id:
                if type_term not in relations[char_type]:
                    relations[char_type][type_term] = {
                        "id": type_id,
                        "targets": {}
                    }
                relations[char_type][type_term]["targets"][target_term] = target_id
    
    return relations

In [221]:
def query_snomed_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=QUERY_PROMPT),
        HumanMessage(content=state['relations'])
    ])
    content = {}
    for q in queries.queries:
        response = snomed_client.search(term=q, limit=2)
        for r in response['items']:
            content[r['fsn']['term']] = r['conceptId']
    return {"search_terms": queries.queries, "snomed_candidates": content}

In [201]:
def predict_attributes(state: AgentState):
    messages = [
        SystemMessage(content=ATTRIBUTE_PROMPT), 
        HumanMessage(content=state['search_terms'][0])
    ]
    attributes = model.with_structured_output(Attributes).invoke(messages)
    return {"target_attributes": attributes.model_dump_json()}

In [280]:
def evaluate_attributes(state: AgentState):
    evals = []
    shortlist = []
    final_candidate = None
    for name, snomed_id in state['snomed_candidates'].items():
        full_concept = snomed_client.search_concept_id(snomed_id)
        relations = extract_relations(full_concept)
        print(relations)
        if relations["INFERRED_RELATIONSHIP"] == {} and relations["STATED_RELATIONSHIP"] == {}:
            print("Relationships not found")
            continue
        score_card = model.with_structured_output(ScoreCard).invoke([
            SystemMessage(content=EVALUATION_PROMPT),
            HumanMessage(content=f"<candidate>\n{name}: {snomed_id}\n\n <predicted_attributes>\n{state['target_attributes']}\n\n<retrieved_relationships>\n{relations}")
        ])
        if score_card.score >= 4:
            shortlist.append(score_card)
        evals.append(score_card)

    top_scores = [scorecard for scorecard in shortlist if scorecard.score == 5]
    
    if len(top_scores) == 1:
        final_candidate = top_scores[0]

    return {"evals": evals, "shortlist": shortlist, 'final_candidate': final_candidate}

In [255]:
def refine_shortlist(state: AgentState):
    print("To implement")
    state['revision_number'] += 1
    pass

In [258]:
def should_refine(state: AgentState):
    if state['final_candidate'] is not None:
        return END
    else:
        return "refine"

In [281]:
builder = StateGraph(AgentState)

In [282]:
builder.add_node("search", query_snomed_node)
builder.add_node("planner", predict_attributes)
builder.add_node("evaluator", evaluate_attributes)
builder.add_node("refine", refine_shortlist)

In [283]:
builder.add_edge("search", "planner")
builder.add_edge("planner", "evaluator")
builder.add_conditional_edges("evaluator", should_refine, {END: END, "refine": "refine"})

In [284]:
builder.set_entry_point("search")

In [285]:
graph = builder.compile()

In [83]:
from IPython.display import Image

Image(graph.get_graph().draw_png())

ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.

In [267]:
example_relation = """{
  'node_1': 'fracture',
  'node_2': 'left femur',
  'edge': 'of'
}"""

In [286]:
output = graph.invoke({
    'relations': example_relation,
    'max_revisions': 2,
    'revision_number': 1,
})

https://snowstorm-training.snomedtools.org/snowstorm/snomed-ct/browser/MAIN%2F2020-07-31/concepts/15911931000119102
{'INFERRED_RELATIONSHIP': {'Is a': {'id': '116680003', 'targets': {'Lesion of left thigh bone': '16302031000119101', 'Open fracture of head of femur': '41191003', 'Open wound of left hip region': '10876711000119109', 'Open wound of left thigh': '10877231000119107'}}, 'Finding site': {'id': '363698007', 'targets': {'Structure of head of left femur': '773966005'}}, 'Associated morphology': {'id': '116676008', 'targets': {'Fracture, open': '52329006'}}}, 'STATED_RELATIONSHIP': {}}
https://snowstorm-training.snomedtools.org/snowstorm/snomed-ct/browser/MAIN%2F2020-07-31/concepts/15912331000119106
{'INFERRED_RELATIONSHIP': {'Is a': {'id': '116680003', 'targets': {'Closed fracture of head of femur': '208526001', 'Injury of left leg': '11865081000119107', 'Lesion of left thigh bone': '16302031000119101'}}, 'Finding site': {'id': '363698007', 'targets': {'Structure of head of left

In [287]:
output

{'relations': "{\n  'node_1': 'fracture',\n  'node_2': 'left femur',\n  'edge': 'of'\n}",
 'search_terms': ['left femur fracture',
  'fracture of left femur',
  'left femoral fracture'],
 'snomed_candidates': {'Open fracture of head of left femur (disorder)': '15911931000119102',
  'Closed fracture of head of left femur (disorder)': '15912331000119106'},
 'target_attributes': '{"finding_site":"Femur","morphology":"Fracture"}',
 'evals': [ScoreCard(candidate_term='Open fracture of head of left femur (disorder)', snomed_id='15911931000119102', score=4, reasoning="- The predicted finding site 'Femur' is closely related to the retrieved finding site 'Structure of head of left femur', which is appropriate for the candidate term.\n- The predicted morphology 'Fracture' is closely related to the retrieved morphology 'Fracture, open', which is consistent with the candidate term.\n- The inferred relationships include relevant 'Is a' targets such as 'Open fracture of head of femur', which support

In [288]:
output['final_candidate']

ScoreCard(candidate_term='Closed fracture of head of left femur (disorder)', snomed_id='15912331000119106', score=5, reasoning="- The predicted finding site 'Femur' is closely related to the retrieved finding site 'Structure of head of left femur', which is specific and appropriate for the candidate term.\n- The predicted morphology 'Fracture' matches the retrieved associated morphology 'Fracture, closed', indicating a precise match.\n- The inferred 'Is a' relationships, such as 'Closed fracture of head of femur', are consistent with the candidate term, supporting its appropriateness.\n- Although there are no stated relationships, the inferred relationships are strong and align well with the predicted attributes, justifying a high score.")